# Kaggle Multi-Model Benchmark + FastAPI Lab

Attach your trained models and a compatible CSV using **Add Input**. Run the cell, then select models, load, check compatibility, benchmark, and query the API.

In [ ]:
# KAGGLE MODEL BENCHMARK + FASTAPI LAB (single cell)
# 1. Kaggle -> Add Input -> attach trained model artifacts and a matching CSV.
# 2. Run this cell. Choose model(s), CSV, task and target. Click Load, Check, Benchmark, Query.
# 3. Only load model files you trust; joblib/pickle deserialization executes Python.
import json, time
from pathlib import Path
import numpy as np
import pandas as pd
import joblib, torch, xgboost as xgb
import ipywidgets as w
from IPython.display import display, clear_output
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, r2_score
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel

ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('.')
OUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
OUT.mkdir(exist_ok=True, parents=True)
MODELS = {}
SUFFIXES = {'.pkl', '.pickle', '.joblib', '.json', '.ubj', '.pt', '.pth', '.jit', '.ts'}
SKIP_NAMES = {'metadata.json', 'dataset-metadata.json', 'kernel-metadata.json'}

# Filenames are only hints; we also verify each loaded object's capabilities.
def category(p):
    s = str(p).lower()
    if p.name.lower() in SKIP_NAMES or p.suffix.lower() not in SUFFIXES: return 'ignore'
    if any(k in p.stem.lower() for k in ('preprocessor', 'preprocess', 'scaler', 'encoder', 'tokenizer', 'vectorizer', 'transformer', 'imputer')):
        return 'preprocessor'
    if p.suffix.lower() == '.pth': return 'checkpoint (requires architecture)'
    return 'candidate model'

def scan():
    found = sorted((p for p in ROOT.rglob('*') if p.is_file()), key=str)
    models = [p for p in found if category(p) == 'candidate model']
    other = [p for p in found if category(p) not in ('candidate model', 'ignore')]
    csvs = [p for p in found if p.suffix.lower() == '.csv']
    return models, other, csvs

model_ui = w.SelectMultiple(description='Models', rows=9, layout=w.Layout(width='99%'), style={'description_width':'70px'})
csv_ui = w.Dropdown(description='Dataset', layout=w.Layout(width='99%'), style={'description_width':'70px'})
target_ui = w.Dropdown(description='Target')
task_ui = w.Dropdown(description='Task', options=['classification', 'regression'])
show_other_ui = w.Checkbox(description='Show preprocessors/checkpoints (advanced)', value=False, indent=False)
refresh_btn = w.Button(description='Refresh inputs')
load_btn = w.Button(description='Load models', button_style='success')
check_btn = w.Button(description='Check compatibility')
bench_btn = w.Button(description='Benchmark', button_style='primary')
query_btn = w.Button(description='Query FastAPI')
log = w.Output()

# Preserve full paths as values: same basenames in different Kaggle versions are distinct.
def refresh(_=None):
    candidates, other, csvs = scan()
    listed = candidates + (other if show_other_ui.value else [])
    model_ui.options = [(f'{category(p)} | {p}', str(p)) for p in listed]
    csv_ui.options = [(str(p), str(p)) for p in csvs]
    set_targets()
    with log:
        clear_output()
        print(f'Found {len(candidates)} candidate models, {len(other)} preprocessors/checkpoints, {len(csvs)} CSV datasets.')
        if other:
            print('Excluded by default (toggle Advanced to inspect):')
            for p in other[:15]: print(' -', p)
        if not candidates: print('No candidate models. Attach trained model artifacts with Kaggle Add Input.')
        if not csvs: print('No CSV datasets. Attach a CSV with Kaggle Add Input.')

def set_targets(_=None):
    try:
        cols = pd.read_csv(csv_ui.value, nrows=1).columns.tolist()
        target_ui.options = cols
        if cols: target_ui.value = cols[-1]
    except Exception:
        target_ui.options = []

def load_one(path):
    p = Path(path); suffix = p.suffix.lower()
    if suffix in ('.pkl', '.pickle', '.joblib'):
        obj = joblib.load(path)  # trusted files only
        kind = 'python'
    elif suffix in ('.json', '.ubj'):
        obj = xgb.Booster(); obj.load_model(str(p)); kind = 'xgboost-booster'
    elif suffix in ('.pt', '.jit', '.ts'):
        obj = torch.jit.load(str(p), map_location='cpu'); obj.eval(); kind = 'torchscript'
    else:
        raise ValueError('Not directly loadable: .pth/state_dict needs model class; select exported TorchScript instead')
    if kind == 'python' and not callable(getattr(obj, 'predict', None)):
        raise ValueError(f'{type(obj).__name__} has no predict(): likely a preprocessor or checkpoint, not a prediction model')
    return obj, kind

def load(_=None):
    MODELS.clear()
    with log:
        clear_output()
        for path in model_ui.value:
            try:
                obj, kind = load_one(path)
                # Unique stable identifier preserves basename and version differences.
                name = Path(path).stem
                n = 2
                while name in MODELS:
                    name = f'{Path(path).stem}_{n}'; n += 1
                MODELS[name] = {'object':obj, 'kind':kind, 'path':path}
                print('READY:', name, '|', type(obj).__name__, '|', path)
            except Exception as e:
                print('SKIPPED:', path, '|', e)
        print('Prediction models loaded:', len(MODELS))

def dataset():
    if not csv_ui.value or not target_ui.value: raise ValueError('Choose CSV and target column')
    df = pd.read_csv(csv_ui.value).dropna(subset=[target_ui.value])
    if df.empty: raise ValueError('No labeled rows')
    return df.drop(columns=[target_ui.value]), df[target_ui.value]

def prepare(name, X):
    obj = MODELS[name]['object']; kind = MODELS[name]['kind']
    expected = None
    if kind == 'python' and hasattr(obj, 'feature_names_in_'):
        expected = list(obj.feature_names_in_)
    elif kind == 'xgboost-booster' and obj.feature_names:
        expected = list(obj.feature_names)
    if expected is not None:
        missing = [col for col in expected if col not in X.columns]
        if missing:
            raise ValueError(f'Missing required input columns: {missing}. CSV has: {list(X.columns)}')
        X = X.loc[:, expected]
    return X

def predict(name, X):
    item = MODELS[name]; obj = item['object']; kind = item['kind']
    X = prepare(name, X)
    if kind == 'python':
        return np.asarray(obj.predict(X)).reshape(-1)
    if kind == 'xgboost-booster':
        raw = np.asarray(obj.predict(xgb.DMatrix(X)))
    else:
        with torch.no_grad():
            raw = obj(torch.as_tensor(X.to_numpy(dtype=np.float32)))
            if isinstance(raw, (list, tuple)): raw = raw[0]
            raw = raw.detach().cpu().numpy()
    if task_ui.value == 'regression': return raw.reshape(-1)
    if raw.ndim == 2 and raw.shape[1] > 1: return raw.argmax(axis=1)
    # Binary predictions here assume model outputs probabilities, not logits.
    return (raw.reshape(-1) >= 0.5).astype(int)

def check(_=None):
    with log:
        clear_output()
        try:
            X, y = dataset()
            print('Dataset:', csv_ui.value, '| rows:', len(X))
            print('Features:', list(X.columns), '| target:', target_ui.value)
            for name in MODELS:
                try:
                    prepared = prepare(name, X.iloc[:1])
                    pred = predict(name, prepared)
                    print('PASS:', name, '| one prediction:', np.asarray(pred).tolist())
                except Exception as e:
                    print('FAIL:', name, '|', e)
        except Exception as e: print('Dataset error:', e)

def benchmark():
    if not MODELS: raise ValueError('Load actual prediction models first')
    X, y = dataset(); rows = []
    for name, item in MODELS.items():
        try:
            predict(name, X.iloc[:min(len(X), 2)])
            timings = []
            for _ in range(5):
                start = time.perf_counter(); pred = predict(name, X)
                timings.append((time.perf_counter()-start)*1000)
            row = dict(model=name, type=item['kind'], rows=len(X),
                       latency_ms=round(float(np.median(timings)), 3),
                       artifact_kb=round(Path(item['path']).stat().st_size/1024, 1))
            if task_ui.value == 'classification':
                row.update(accuracy=accuracy_score(y, pred),
                           f1=f1_score(y, pred, average='weighted', zero_division=0))
            else:
                row.update(rmse=float(np.sqrt(mean_squared_error(y, pred))), r2=r2_score(y, pred))
            rows.append(row)
        except Exception as e: rows.append(dict(model=name, error=str(e)))
    board = pd.DataFrame(rows)
    key = 'f1' if task_ui.value == 'classification' else 'rmse'
    if key in board: board = board.sort_values(key, ascending=(key == 'rmse'), na_position='last')
    board.to_csv(OUT/'benchmark_leaderboard.csv', index=False)
    board.to_json(OUT/'benchmark_leaderboard.json', orient='records', indent=2)
    return board

app = FastAPI(title='Kaggle ML Benchmark Serving Lab')
class Request(BaseModel):
    instances: list[dict]

@app.get('/health')
def health(): return {'ok': True, 'loaded':len(MODELS)}

@app.get('/models')
def models(): return [{'name':n, 'kind':v['kind'], 'path':v['path']} for n,v in MODELS.items()]

@app.post('/predict/{name}')
def predict_one(name: str, request: Request):
    if name not in MODELS: raise HTTPException(404, 'Model not found')
    try: return {'model':name, 'predictions':predict(name, pd.DataFrame(request.instances)).tolist()}
    except Exception as e: raise HTTPException(422, str(e))

@app.post('/predict-all')
def predict_all(request: Request):
    X = pd.DataFrame(request.instances); results = {}
    for name in MODELS:
        try: results[name] = predict(name, X).tolist()
        except Exception as e: results[name] = {'error':str(e)}
    return {'results':results}

@app.post('/benchmark')
def benchmark_route():
    try: return {'leaderboard':json.loads(benchmark().to_json(orient='records'))}
    except Exception as e: raise HTTPException(422, str(e))

client = TestClient(app)  # Local HTTP simulation, no exposed Kaggle port.

def show_benchmark(_=None):
    with log:
        clear_output()
        try:
            display(benchmark()); print('Saved:', OUT/'benchmark_leaderboard.csv')
        except Exception as e: print('Benchmark error:', e)

def query(_=None):
    with log:
        clear_output()
        try:
            X, _ = dataset()
            payload = {'instances':json.loads(X.iloc[:1].to_json(orient='records'))}
            print('GET /health:', client.get('/health').json())
            print('GET /models:', json.dumps(client.get('/models').json(), indent=2))
            res = client.post('/predict-all', json=payload)
            print('POST /predict-all:', res.status_code, json.dumps(res.json(), indent=2))
        except Exception as e: print('API query error:', e)

refresh_btn.on_click(refresh)
show_other_ui.observe(refresh, names='value')
csv_ui.observe(set_targets, names='value')
load_btn.on_click(load)
check_btn.on_click(check)
bench_btn.on_click(show_benchmark)
query_btn.on_click(query)

display(w.HTML('<h3>Kaggle Multi-model Benchmark + FastAPI Lab</h3>'))
display(show_other_ui, model_ui, csv_ui, target_ui, task_ui)
display(w.HBox([refresh_btn, load_btn, check_btn, bench_btn, query_btn]), log)
refresh()
print('Ready. Select genuine trained model artifacts, then Load -> Check compatibility -> Benchmark -> Query FastAPI.')
